# Find rare classes threshold

In [1]:
from pathlib import Path
import pandas as pd
import sys

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS
from models_helper import per_class_learning_curve
from sklearn.ensemble import RandomForestClassifier
from _settings import SEED

In [2]:
DATA_DIR = "../3_7_split dataset to datasets"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train.parquet")
df_val = pd.read_parquet(Path(DATA_DIR) / "dataset_validation.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,mat_naturstein,mat_putz,mat_stahl,mat_zement,horizontal_elements_above,horizontal_elements_below,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing
0,0.0,-3.199996,-0.07,0.33,0.000000,0.0,0.33,3.199996,0.07,0.212121,...,0.0,0.0,0.0,0.0,79.0,0.0,IfcCovering,FLOORING,false,unknown
1,0.0,0.000000,-0.05,6.91,5.969983,0.0,6.91,5.969983,0.05,0.007236,...,0.0,0.0,0.0,0.0,88.0,0.0,IfcCovering,FLOORING,false,unknown
2,0.0,-3.355000,-0.05,4.36,1.514983,0.0,4.36,4.869983,0.05,0.011468,...,0.0,0.0,0.0,0.0,91.0,0.0,IfcCovering,FLOORING,false,unknown
3,0.0,0.000000,-0.02,6.81,13.740000,0.0,6.81,13.740000,0.02,0.002937,...,0.0,0.0,0.0,0.0,104.0,0.0,IfcCovering,FLOORING,false,unknown
4,0.0,0.000000,-0.07,6.81,7.590000,0.0,6.81,7.590000,0.07,0.010279,...,0.0,0.0,0.0,0.0,88.0,0.0,IfcCovering,FLOORING,false,unknown


## 1. Class distribution analysis

In [3]:
# show main labels
LABEL_COLS = [c for c in df_train.columns if c.startswith("label_")]

# show class distribution for each label column
for col in LABEL_COLS:
    counts = df_train[col].value_counts()
    print(f"\n{col} ({len(counts)} classes):")
    print(counts.to_string())


label_ifc_entity (12 classes):
label_ifc_entity
IfcWall           14167
IfcRailing         7000
IfcSpace           3922
IfcSlab            1632
IfcWindow          1339
IfcPlate            954
IfcDoor             825
IfcRoof             815
IfcCovering         349
IfcFurniture        171
IfcStairFlight      135
IfcColumn           127

label_predefined_type (18 classes):
label_predefined_type
SOLIDWALL       9904
GUARDRAIL       7000
INTERNAL        3692
NOTDEFINED      1933
PLUMBINGWALL    1450
WINDOW          1339
FLOOR           1285
SHEET            954
DOOR             822
FLAT_ROOF        815
PARTITIONING     795
PARAPET          395
FLOORING         349
GFA              230
BASESLAB         219
ROOF             128
MOVABLE          123
GATE               3

label_is_external (3 classes):
label_is_external
true       16918
false      10290
unknown     4228

label_load_bearing (3 classes):
label_load_bearing
unknown    13785
true       11097
false       6554


## 2. Per-class learning curve analysis

In [4]:
# create feature matrices
X_train = df_train[ALL_FEATURE_KEYS].values
X_val = df_val[ALL_FEATURE_KEYS].values

# baseline model for learning curve analysis, RF can handle small sample sizes
model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

# set candidate threshold for rare classes
CANDIDATE_THRESHOLD = 1000

# analyze each label column for rare classes and plot learning curves
for col in LABEL_COLS:
    y_train = df_train[col].values
    y_val = df_val[col].values
    counts = df_train[col].value_counts()
    rare_candidates = counts[counts < CANDIDATE_THRESHOLD].index.tolist()

    if not rare_candidates:
        print(f"{col}: No rare classes detected")
        continue

    print(f"\n{col} - {len(rare_candidates)} rare candidates:")
    for cls in rare_candidates:
        print(f"  '{cls}': {(y_train == cls).sum()} Train / {(y_val == cls).sum()} Val Samples")

    per_class_learning_curve(
        X_train, y_train, model,
        class_labels=rare_candidates,
        label_name=col,
        X_val=X_val, y_val=y_val,
        threshold=200,
        save=f"learning_curve_{col}",
        chapter="implementation"
    )


label_ifc_entity - 7 rare candidates:
  'IfcPlate': 954 Train / 230 Val Samples
  'IfcDoor': 825 Train / 169 Val Samples
  'IfcRoof': 815 Train / 126 Val Samples
  'IfcCovering': 349 Train / 125 Val Samples
  'IfcFurniture': 171 Train / 98 Val Samples
  'IfcStairFlight': 135 Train / 55 Val Samples
  'IfcColumn': 127 Train / 19 Val Samples
figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/learning_curve_label_ifc_entity.svg



label_predefined_type - 11 rare candidates:
  'SHEET': 954 Train / 230 Val Samples
  'DOOR': 822 Train / 167 Val Samples
  'FLAT_ROOF': 815 Train / 126 Val Samples
  'PARTITIONING': 795 Train / 93 Val Samples
  'PARAPET': 395 Train / 22 Val Samples
  'FLOORING': 349 Train / 125 Val Samples
  'GFA': 230 Train / 51 Val Samples
  'BASESLAB': 219 Train / 92 Val Samples
  'ROOF': 128 Train / 31 Val Samples
  'MOVABLE': 123 Train / 4 Val Samples
  'GATE': 3 Train / 2 Val Samples
figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/learning_curve_label_predefined_type.svg


label_is_external: No rare classes detected
label_load_bearing: No rare classes detected


## 3. Set threshold and filter rare classes

In [5]:
# set threshold for rare classes based on learning curve analysis
# optimal is between 200 wit a f1 macro score over 0.4 to keep as much as classes in the dataset, while still ensuring learnability.
RARE_CLASS_THRESHOLD = 200

print(f"Threshold for rare classes: {RARE_CLASS_THRESHOLD} Samples\n")

print("Classes below the defined threshold:\n")
for col in LABEL_COLS:
    counts = df_train[col].value_counts()
    rare = counts[counts < RARE_CLASS_THRESHOLD]
    
    if not rare.empty:
        print(f"{col}: {len(rare)}")

Threshold for rare classes: 200 Samples

Classes below the defined threshold:

label_ifc_entity: 3
label_predefined_type: 3
